# 在 HPC 集群上安装 OLLAMA

参考： https://rc.northeastern.edu/2025/03/26/installing-ollama-on-the-hpc-cluster/


## 在终端中从 OLLAMA 容器创建一个 .sif 文件
容器允许可移植的软件执行。OLLAMA 在 Docker Hub 上有一个官方容器。（Docker Hub 是一个容器存储库）。

在集群上，我们使用 Singularity 来运行容器。我们可以使用以下命令在集群上设置 OLLAMA 容器（以便可以使用 Singularity 运行它）。



In [ ]:
cd /scratch/$USER
mkdir ollama_models_scratch # to store the LLM models in /scratch directory

module load singularity/3.10.3
singularity pull ollama.sif docker://ollama/ollama


现在，您可以在此终端中使用 OLLAMA



In [ ]:
export SINGULARITYENV_OLLAMA_MODELS=/scratch/$USER/ollama_models_scratch
singularity run --nv -B "/home:/home,/work:/work,/scratch:/scratch" ollama.sif


## 在另一个终端运行特定模型
首先，在 Jupyterlab 中启动另一个终端。然后，您可以使用以下命令加载并运行模型（例如llama3.2:latest或deepseek-r1:1.5b）：

In [ ]:
cd /scratch/$USER
module load singularity/3.10.3
unset http_proxy https_proxy
singularity run --nv -B “/home:/home,/work:/work,/scratch:/scratch” ollama.sif run deepseek-r1:1.5b



nohp singularity run --nv \
  --env OLLAMA_HOST=0.0.0.0:11434 \
  --env OLLAMA_KEEP_ALIVE=10m \
  -B ~/.ollama/models:/scratch/username/.ollama/models \
  /path/ollama.sif \
  serve  > /path/ollama_21.log 2>&1 &

这将创建一个命令行提示符来与模型交互。

## 从python访问模型
Jupyterlab 还可以用于在集群上运行可用的 LLM 模型进行开发。请注意，您需要取消设置 http 和 https 代理变量。

首先，在 Jupyterlab 的新终端会话中，运行以下命令：

In [ ]:
pip install ollama


然后，在新的 Jupyter 笔记本中，粘贴以下代码并运行：

In [ ]:
import os
os.environ.pop("http_proxy", None)
os.environ.pop("https_proxy", None)

import ollama
response = ollama.chat(
model='deepseek-r1:1.5b',
messages=[{'role': 'user', "content": "Your question here"}]
)
print(response['message']['content'])


## 其他方式



In [ ]:
singularity instance start --nv \
  --env OLLAMA_HOST=0.0.0.0:11434 \
  --env OLLAMA_KEEP_ALIVE=10m \
  -B ~/.ollama/models:/scratch/username/.ollama/models \
  /slurm/home/yrd/shaolab/username/projects/homaf/singularity_template/ollama.sif \
  ollama-instance 

# 然后手动启动
singularity shell instance://ollama-instance

ollama serve